# L35 · RLHF 全流程：从人类反馈到策略优化

**学习目标**
- 串起完整 RLHF 三步流水线：SFT → 奖励模型(RM) → PPO 强化
- 理解「人类反馈」如何最终变成「模型行为」
- 用 numpy 投影版跑通端到端，看模型从「通用」变「对齐」

**前置依赖**：L31（SFT）、L32（RM）、L34（PPO）  
**预计时长**：60 分钟  
**技术栈**：`numpy`、`matplotlib`（离线可运行）

---

## 概念讲解：RLHF = 用人类喜好「雕刻」模型

RLHF（Reinforcement Learning from Human Feedback）是 ChatGPT 的成名绝技，三步：

1. **SFT**：用示范数据教模型「会说人话」（L31）
2. **RM**：训练奖励模型，学会「人类偏好哪个回答」（L32）
3. **PPO**：用奖励模型当「零食」，强化模型产出更受人类喜欢的回答（L34）

本课把三步**串成一条流水线**，让你看清「人类反馈如何一路变成模型行为」。

## 第一步：SFT 阶段（先学会基本格式）

In [ ]:
import numpy as np
np.random.seed(0)
# 指令特征(4维) → 回答特征(2维)，SFT 学这个映射
X = np.random.randn(50, 4)
W_true = np.array([[1,0,0,0],[0,1,0,0]]).T
Y = X @ W_true + np.random.normal(0,0.1,(50,2))
W = np.random.randn(4,2)*0.1
for _ in range(200):
    pred = X @ W
    W -= 0.05 * (2 * X.T @ (pred - Y) / len(X))
print("✅ SFT 完成（模型已会基本输出格式）")

## 第二步：训练奖励模型（RM）

In [ ]:
good = np.random.randn(60,2)+1.0
bad = np.random.randn(60,2)-1.0
rw = np.random.randn(2)*0.1
for _ in range(300):
    g = (rw @ good.mean(0) - rw @ bad.mean(0))
    rw += 0.1 * (1/(1+np.exp(-g)) - 1) * (good.mean(0) - bad.mean(0))
print("✅ 奖励模型训练完成（学会偏好好回答）")

## 第三步：PPO 强化 —— 用 RM 当奖励微调 W

In [ ]:
def sigmoid(x): return 1/(1+np.exp(-x))
ref_W = W.copy()
r_history = []
for step in range(200):
    # 对每条指令，模型产出回答，RM 打分
    out = X @ W
    score = rw @ out.mean(0) - 0.1*np.sum((W-ref_W)**2)   # 奖励 - KL 惩罚
    r_history.append(score)
    grad = rw * (X.mean(0)[:,None]) - 0.2*(W-ref_W)
    W += 0.01 * grad
print("✅ PPO 对齐完成（模型被奖励模型推向人类偏好）")

# 🎯 AHA 顿悟单元格：端到端 RLHF · 模型被「人类偏好」重塑

运行下面代码。你会看到一张**奖励曲线**：PPO 阶段，模型在「奖励模型」引导下，
平均奖励持续上升（同时受 KL 约束不崩）。这就是「人类反馈」一路变成「模型行为」的全过程。

> 你刚跑通的，就是 ChatGPT 训练流水线的完整缩略版。从一行指令到对齐助手，三步走完。

In [ ]:
# ===== 运行我！看 RLHF 端到端奖励上升 =====
import matplotlib.pyplot as plt
sm = np.convolve(r_history, np.ones(10)/10, mode="valid")
plt.figure(figsize=(7,4))
plt.plot(sm, color="#d62728", lw=2)
plt.title("RLHF 全流程：PPO 阶段平均奖励（含 KL 约束）")
plt.xlabel("训练步"); plt.ylabel("奖励")
plt.grid(alpha=0.3); plt.show()
print(f"  🎯 奖励从 {r_history[0]:.3f} 提升到 {r_history[-1]:.3f}")
print("  ✨ SFT → RM → PPO 三段串联完成，你亲手跑通了 ChatGPT 的对齐流水线！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：三步因果链（SFT 给底子 → RM 定标准 → PPO 强化）；KL 惩罚防跑偏。  
**易错点**：各阶段随机种子与独立训练需衔接；本方案各阶段用简化投影，非真实 Transformer，但流水线逻辑一致。  
**AHA 机制**：端到端奖励曲线，强「人类反馈变成模型行为」的完整认知闭环，阶段六情绪高点。  
**衔接**：L36 RL 项目（更复杂环境实战）；L39 后训练管线工程化。  
**依赖**：`pip install numpy matplotlib`。  
**真 LLM 路径**：真实 RLHF 用 `transformers` + `TRL (PPOTrainer/SFTTrainer)` + 偏好数据集，需 GPU；本演示为原理投影。

# 📚 作业 / 下一步

1. 把 KL 惩罚系数 `0.1` 改成 `0.5`，看奖励上升是否变缓（约束更强）。
2. 搜索「InstructGPT 论文」理解真实 RLHF 数据构造。
3. 下一课 **L36 强化学习项目：训练一个会博弈的 AI** —— 用 PPO 思想实战一个井字棋/猜数字对手。